# Analyse des hyperparamètres — RBF, MLP & SVM

Ce notebook explore l'impact des hyperparamètres sur les performances des modèles RBF, MLP et SVM.

**Hyperparamètres étudiés :**
- **RBF** : `n_centers` (nombre de centres), `lr` (taux d'apprentissage), `epochs`
- **MLP** : `lr`, `epochs`, architecture des couches cachées
- **SVM** : `C` (régularisation), type de noyau (`linéaire` / `RBF` / `polynomial`)

**Objectif** : trouver la configuration optimale pour maximiser l'accuracy sur le jeu de test.

In [ ]:
import os, sys, subprocess, time
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = os.path.exists('/content')

if IN_COLAB:
    BASE_DIR = '/content/VisionAI'

    if not os.path.exists(BASE_DIR):
        subprocess.run(['git', 'clone', '-b', 'ML_finition',
                        'https://github.com/SINCER-Ali/VisionAI.git', BASE_DIR], check=True)
    else:
        subprocess.run(['git', '-C', BASE_DIR, 'pull'], capture_output=True)

    if subprocess.run(['which', 'cargo'], capture_output=True).returncode != 0:
        subprocess.run(
            'curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable',
            shell=True, check=True
        )

    cargo_bin = '/root/.cargo/bin'
    os.environ['PATH'] = cargo_bin + ':' + os.environ.get('PATH', '')
    subprocess.run(['pip', 'install', 'maturin', '-q'], check=True)

    print('Build du wheel vision_ai...')
    result = subprocess.run(
        ['maturin', 'build', '--release', '-i', sys.executable],
        cwd=f'{BASE_DIR}/python_binding',
        env={**os.environ, 'PATH': cargo_bin + ':' + os.environ.get('PATH', '')},
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f'Build échoué (code {result.returncode})')

    wheel_dir = f'{BASE_DIR}/target/wheels'
    wheels = [f for f in os.listdir(wheel_dir) if f.endswith('.whl')]
    subprocess.run(['pip', 'install', os.path.join(wheel_dir, wheels[-1]), '--force-reinstall', '-q'], check=True)
    print('Binding installé ✓')

else:
    BASE_DIR = os.path.dirname(os.path.abspath('.'))

DATASET_DIR = os.path.join(BASE_DIR, 'datasets')

import vision_ai
CLASSES = ['aucun', 'humain', 'animal']
print('vision_ai importé ✓')

In [ ]:
# === Dataset → .npy ===
# - En LOCAL : on utilise les .npy déjà générés par preprocess_dataset.py (rien à télécharger).
# - Sur COLAB : on monte le Drive et on convertit les images de IMG_dataset en .npy.
import os
_need = not os.path.exists(os.path.join(DATASET_DIR, 'X_train.npy'))

if _need and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # "IMG_dataset" est dans "Partagés avec moi" → il faut un RACCOURCI vers Mon Drive
    # (Drive web → clic droit sur IMG_dataset → Organiser → Ajouter un raccourci à Drive).
    # On essaie les emplacements probables du raccourci :
    _candidats = [
        '/content/drive/MyDrive/new_dataset/IMG_dataset',
        '/content/drive/MyDrive/IMG_dataset',
    ]
    IMAGES_DIR = next((p for p in _candidats if os.path.isdir(p)), None)
    if IMAGES_DIR is None:
        raise FileNotFoundError(
            "IMG_dataset introuvable dans Mon Drive. Vérifie le chemin exact dans le panneau "
            "Fichiers de Colab et ajoute-le à _candidats. (As-tu bien ajouté le raccourci ?)"
        )
    print(f'Dossier images : {IMAGES_DIR}')

    import numpy as np
    from PIL import Image
    from sklearn.model_selection import train_test_split

    LABEL_OF = {'aucun': 0, 'humain': 1, 'animal': 2, 'animaux': 2}   # nom_normalisé -> label
    X, y = [], []
    for classe in sorted(os.listdir(IMAGES_DIR)):
        dossier = os.path.join(IMAGES_DIR, classe)
        if not os.path.isdir(dossier):
            continue
        key = classe.strip().lower()
        if key not in LABEL_OF:
            print(f'  ⚠️ dossier ignoré (nom inconnu) : {classe}')
            continue
        label = LABEL_OF[key]
        fichiers = os.listdir(dossier)
        print(f'  {classe} -> label {label} : {len(fichiers)} images')
        for f in fichiers:
            try:
                img = Image.open(os.path.join(dossier, f)).convert('RGB').resize((64, 64))
                X.append(np.array(img, dtype='float32').flatten() / 255.0)
                y.append(label)
            except Exception:
                pass

    X = np.array(X, dtype='float32'); y = np.array(y)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    os.makedirs(DATASET_DIR, exist_ok=True)
    np.save(os.path.join(DATASET_DIR, 'X_train.npy'), Xtr)
    np.save(os.path.join(DATASET_DIR, 'X_test.npy'),  Xte)
    np.save(os.path.join(DATASET_DIR, 'y_train.npy'), ytr)
    np.save(os.path.join(DATASET_DIR, 'y_test.npy'),  yte)
    print(f'✅ {len(X)} images converties → .npy')

elif _need:
    raise FileNotFoundError(
        "Aucun .npy trouvé en local. Lance d'abord :  python preprocess_dataset.py"
    )
else:
    print('✅ Dataset .npy déjà présent — rien à régénérer.')

## 1. Chargement du dataset

In [ ]:
X_train = np.load(os.path.join(DATASET_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(DATASET_DIR, 'y_train.npy'))
X_test  = np.load(os.path.join(DATASET_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(DATASET_DIR, 'y_test.npy'))

# Sous-échantillonnage (1 pixel sur 4) pour réduire la dim RBF
STEP = 4
X_train_r = X_train[:, ::STEP]
X_test_r  = X_test[:, ::STEP]
DIM_RBF   = X_train_r.shape[1]
INPUT_SIZE = X_train.shape[1]

inputs_train_r = X_train_r.tolist()
inputs_test_r  = X_test_r.tolist()
inputs_train   = X_train.tolist()
inputs_test    = X_test.tolist()

def one_hot(labels, n=3):
    return [[1.0 if int(l)==i else 0.0 for i in range(n)] for l in labels]

targets_train = one_hot(y_train)

def accuracy_fn(predict_fn, inputs, labels):
    preds = [predict_fn(x) for x in inputs]
    y_pred = [p.index(max(p)) for p in preds]
    return sum(p == int(t) for p, t in zip(y_pred, labels)) / len(labels)

print(f'Train : {X_train.shape}  |  Test : {X_test.shape}')
print(f'Dim RBF (réduite) : {DIM_RBF}  |  Dim MLP (plein) : {INPUT_SIZE}')

## 2. Impact du nombre de centres — RBF

In [ ]:
n_centers_list = [5, 10, 20, 30, 50]
accs_centers   = []
times_centers  = []

for n in n_centers_list:
    t0 = time.time()
    rbf = vision_ai.PyRBF(DIM_RBF, 3, n_centers=n, sigma=1.0)
    rbf.init_centers_random(inputs_train_r)
    rbf.train(inputs_train_r, targets_train, lr=0.01, epochs=80, regression=False)
    elapsed = time.time() - t0
    acc = accuracy_fn(rbf.predict, inputs_test_r, y_test)
    accs_centers.append(acc * 100)
    times_centers.append(elapsed)
    print(f'n_centers={n:3d}  →  acc={acc*100:.1f}%  ({elapsed:.1f}s)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(n_centers_list, accs_centers, 'o-', color='darkorange')
axes[0].set_xlabel('Nombre de centres (n_centers)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('RBF — Accuracy vs n_centers')
axes[0].grid(True, alpha=0.3)

axes[1].plot(n_centers_list, times_centers, 's-', color='steelblue')
axes[1].set_xlabel('Nombre de centres (n_centers)')
axes[1].set_ylabel("Temps d'entraînement (s)")
axes[1].set_title("RBF — Temps vs n_centers")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_idx = accs_centers.index(max(accs_centers))
print(f'\nMeilleur n_centers : {n_centers_list[best_idx]} → {accs_centers[best_idx]:.1f}%')

## 3. Impact du learning rate — RBF

In [ ]:
lr_list    = [0.001, 0.005, 0.01, 0.05, 0.1]
accs_lr_rbf = []

for lr in lr_list:
    rbf = vision_ai.PyRBF(DIM_RBF, 3, n_centers=30, sigma=1.0)
    rbf.init_centers_random(inputs_train_r)
    rbf.train(inputs_train_r, targets_train, lr=lr, epochs=80, regression=False)
    acc = accuracy_fn(rbf.predict, inputs_test_r, y_test)
    accs_lr_rbf.append(acc * 100)
    print(f'lr={lr:.3f}  →  acc={acc*100:.1f}%')

plt.figure(figsize=(7, 4))
plt.semilogx(lr_list, accs_lr_rbf, 'o-', color='darkorange')
plt.xlabel('Learning rate (échelle log)')
plt.ylabel('Accuracy (%)')
plt.title('RBF — Accuracy vs Learning Rate')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_idx = accs_lr_rbf.index(max(accs_lr_rbf))
print(f'\nMeilleur lr pour RBF : {lr_list[best_idx]} → {accs_lr_rbf[best_idx]:.1f}%')

## 4. Impact du nombre d'époques — RBF

In [ ]:
epochs_list   = [20, 50, 100, 200, 300]
accs_ep_rbf   = []

for ep in epochs_list:
    rbf = vision_ai.PyRBF(DIM_RBF, 3, n_centers=30, sigma=1.0)
    rbf.init_centers_random(inputs_train_r)
    rbf.train(inputs_train_r, targets_train, lr=0.01, epochs=ep, regression=False)
    acc = accuracy_fn(rbf.predict, inputs_test_r, y_test)
    accs_ep_rbf.append(acc * 100)
    print(f'epochs={ep:4d}  →  acc={acc*100:.1f}%')

plt.figure(figsize=(7, 4))
plt.plot(epochs_list, accs_ep_rbf, 'o-', color='darkorange')
plt.xlabel('Nombre d\'époques')
plt.ylabel('Accuracy (%)')
plt.title('RBF — Accuracy vs Epochs')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Impact du learning rate — MLP

In [ ]:
lr_list_mlp  = [0.0001, 0.0005, 0.001, 0.005, 0.01]
accs_lr_mlp  = []

for lr in lr_list_mlp:
    mlp = vision_ai.PyMLP([INPUT_SIZE, 64, 3])
    mlp.train(inputs_train, targets_train, learning_rate=lr, epochs=15)
    acc = accuracy_fn(mlp.predict, inputs_test, y_test)
    accs_lr_mlp.append(acc * 100)
    print(f'lr={lr:.4f}  →  acc={acc*100:.1f}%')

plt.figure(figsize=(7, 4))
plt.semilogx(lr_list_mlp, accs_lr_mlp, 'o-', color='seagreen')
plt.xlabel('Learning rate (échelle log)')
plt.ylabel('Accuracy (%)')
plt.title('MLP — Accuracy vs Learning Rate')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_idx = accs_lr_mlp.index(max(accs_lr_mlp))
print(f'\nMeilleur lr pour MLP : {lr_list_mlp[best_idx]} → {accs_lr_mlp[best_idx]:.1f}%')

## 6. Impact de l'architecture MLP (couches cachées)

In [ ]:
architectures = [
    [INPUT_SIZE, 32, 3],
    [INPUT_SIZE, 64, 3],
    [INPUT_SIZE, 128, 3],
    [INPUT_SIZE, 64, 32, 3],
    [INPUT_SIZE, 128, 64, 3],
]
arch_labels = ['32', '64', '128', '64-32', '128-64']
accs_arch   = []

for arch, label in zip(architectures, arch_labels):
    mlp = vision_ai.PyMLP(arch)
    mlp.train(inputs_train, targets_train, learning_rate=0.001, epochs=15)
    acc = accuracy_fn(mlp.predict, inputs_test, y_test)
    accs_arch.append(acc * 100)
    print(f'Architecture {label:8s}  →  acc={acc*100:.1f}%')

plt.figure(figsize=(8, 4))
bars = plt.bar(arch_labels, accs_arch, color='seagreen', width=0.5)
plt.ylim(0, 100)
plt.xlabel('Couches cachées')
plt.ylabel('Accuracy (%)')
plt.title('MLP — Accuracy vs Architecture')
for bar, acc in zip(bars, accs_arch):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{acc:.1f}%', ha='center', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.show()

best_idx = accs_arch.index(max(accs_arch))
print(f'\nMeilleure architecture : {arch_labels[best_idx]} → {accs_arch[best_idx]:.1f}%')

## 7. Impact des hyperparamètres — SVM (C et noyau)

In [ ]:
# --- Impact du paramètre C (SVM linéaire) ---
C_list     = [0.01, 0.1, 1.0, 10.0, 100.0]
accs_C_svm = []

for c in C_list:
    svm = vision_ai.PySVM(c=c, kernel='linear')
    svm.train(inputs_train, targets_train, lr=0.001, epochs=50)
    acc = accuracy_fn(svm.predict, inputs_test, y_test)
    accs_C_svm.append(acc * 100)
    print(f'C={c:7.2f}  →  acc={acc*100:.1f}%')

plt.figure(figsize=(7, 4))
plt.semilogx(C_list, accs_C_svm, 'o-', color='crimson')
plt.xlabel('Paramètre C (échelle log)')
plt.ylabel('Accuracy (%)')
plt.title('SVM linéaire — Accuracy vs C')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_idx = accs_C_svm.index(max(accs_C_svm))
print(f'\nMeilleur C pour SVM : {C_list[best_idx]} → {accs_C_svm[best_idx]:.1f}%')

# --- Comparaison des noyaux ---
# Les SVM à noyau coûtent O(n^2) : on utilise un sous-ensemble et la dim réduite
SUB = 300
sub_inputs  = inputs_train_r[:SUB]
sub_targets = targets_train[:SUB]

kernels       = [('linear', {}), ('rbf', {'gamma': 0.01})]
accs_kernel   = []
kernel_labels = []

for kname, kparams in kernels:
    svm = vision_ai.PySVM(c=1.0, kernel=kname, **kparams)
    svm.train(sub_inputs, sub_targets, lr=0.001, epochs=50)
    acc = accuracy_fn(svm.predict, inputs_test_r, y_test)
    accs_kernel.append(acc * 100)
    kernel_labels.append(kname)
    print(f'noyau={kname:7s}  →  acc={acc*100:.1f}%')

plt.figure(figsize=(7, 4))
bars = plt.bar(kernel_labels, accs_kernel, color=['crimson', 'mediumpurple'], width=0.5)
plt.ylim(0, 100)
plt.ylabel('Accuracy (%)')
plt.title('SVM — Accuracy vs Noyau')
for bar, acc in zip(bars, accs_kernel):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{acc:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

best_idx = accs_kernel.index(max(accs_kernel))
print(f'\nMeilleur noyau pour SVM : {kernel_labels[best_idx]} → {accs_kernel[best_idx]:.1f}%')

## 8. Récapitulatif des meilleurs hyperparamètres

In [ ]:
print('========== RÉCAPITULATIF DES MEILLEURS HYPERPARAMÈTRES ==========')
print()
print('RBF :')
best_nc  = n_centers_list[accs_centers.index(max(accs_centers))]
best_lr_rbf = lr_list[accs_lr_rbf.index(max(accs_lr_rbf))]
best_ep  = epochs_list[accs_ep_rbf.index(max(accs_ep_rbf))]
print(f'  n_centers optimal : {best_nc}')
print(f'  lr optimal        : {best_lr_rbf}')
print(f'  epochs optimal    : {best_ep}')
print()
print('MLP :')
best_lr_mlp  = lr_list_mlp[accs_lr_mlp.index(max(accs_lr_mlp))]
best_arch    = arch_labels[accs_arch.index(max(accs_arch))]
print(f'  lr optimal        : {best_lr_mlp}')
print(f'  architecture      : [{best_arch}]')
print()
print('SVM :')
best_C      = C_list[accs_C_svm.index(max(accs_C_svm))]
best_kernel = kernel_labels[accs_kernel.index(max(accs_kernel))]
print(f'  C optimal         : {best_C}')
print(f'  meilleur noyau    : {best_kernel}')
print()
print('Ces valeurs servent de configuration de référence pour le notebook de comparaison.')